# Create Helmsley Charitable Trust Awards (GRANT PATTERN, method-5 static HTML)

Leona M. and Harry B. Helmsley Charitable Trust grants from the foundation's own WordPress-published grants archive at helmsleytrust.org. One of the largest US private foundations (~$700M/year), with programs across Type 1 Diabetes, Crohn's Disease, Health Sciences, Conservation, Rural Healthcare, Vulnerable Children in Sub-Saharan Africa, and Israel non-sectarian programs.

**Prerequisites:**
- Run `scripts/local/helmsley_to_s3.py` first.

**Data source:** 4 WordPress sitemap files enumerate the full corpus:
- `/grants-sitemap.xml` (1,000 URLs)
- `/grants-sitemap2.xml` (1,000 URLs)
- `/grants-sitemap3.xml` (1,000 URLs)
- `/grants-sitemap4.xml` (93 URLs)
- **TOTAL: 3,093 grant URLs**

Per-grant detail page has a `<div class="grant-info">` block with `<h6>Label</h6><p>Value</p>` sibling pairs exposing Date of Award, Program, Amount, Term of Grant, Project Title. The H1 is the recipient organization name.

**S3 location:** `s3a://openalex-ingest/awards/helmsley/helmsley_grants.parquet`

**Awarding body:**
- funder_id: 4320309446
- display_name: "Leona M. and Harry B. Helmsley Charitable Trust"
- country: US
- ROR: none
- DOI: 10.13039/100007028

**Coverage (smoke test, 10 grants, 2026-05-22):**
- 100% recipient / slug / program / amount / award_year / project_title / term
- 10/10 unique native_grant_id (numeric suffix of slug)
- Amount range $153,120 - $9,761,000 (median $300k)

Full corpus run (~16 min at 0.3s throttle) was running locally at PR-open time; admin re-running on their side will validate end-to-end. Parser logs parse-fails and continues without crashing.

**Programs (visible in `program` field):**
- Type 1 Diabetes
- Crohn's Disease
- Health Sciences
- Conservation
- Rural Healthcare
- Vulnerable Children in Sub-Saharan Africa
- Israel non-sectarian (Israel-focused programs)
- (others)

**Provenance:** `helmsley_grants` (direct from foundation, not an aggregator).

**Priority:** 112 (UCOP 106 / Rita Allen 107 / Schmidt Sciences 108 / NOMIS 109 / Wenner-Gren 110 / KAW 111 are immediately-prior slots in flight; Vilcek at 105 is current main registry head).


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.helmsley_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/helmsley/helmsley_grants.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.helmsley_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.helmsley_raw;


## Step 1.5: Inspect Raw Data, Money Scan, Native Key

Helmsley publishes per-grant USD amounts on every detail page. Currency is hardcoded USD (single-country US foundation).

In [ ]:
%sql
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'helmsley_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'helmsley_raw'
  AND LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
SELECT recipient, slug, program, amount, currency, date_raw, award_year, term,
       SUBSTR(project_title, 1, 150) AS title_preview
FROM openalex.awards.helmsley_raw LIMIT 5;


In [ ]:
%sql
-- Amount distribution sanity check
SELECT
    COUNT(*) AS total_rows,
    COUNT(amount) AS rows_with_amount,
    SUM(CASE WHEN TRY_CAST(amount AS DOUBLE) IS NOT NULL THEN 1 ELSE 0 END) AS rows_parseable_as_number,
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    ROUND(AVG(TRY_CAST(amount AS DOUBLE)), 0) AS avg_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)) / 1e9, 2) AS total_usd_billions
FROM openalex.awards.helmsley_raw;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.helmsley_raw;


In [ ]:
%sql
SELECT award_year, COUNT(*) as grants
FROM openalex.awards.helmsley_raw
WHERE award_year IS NOT NULL
GROUP BY award_year ORDER BY award_year DESC;


In [ ]:
%sql
SELECT program, COUNT(*) as cnt, ROUND(SUM(TRY_CAST(amount AS DOUBLE))/1e6, 1) AS total_usd_m
FROM openalex.awards.helmsley_raw
GROUP BY program ORDER BY cnt DESC LIMIT 20;


In [ ]:
%sql
SELECT recipient, COUNT(*) as cnt, ROUND(SUM(TRY_CAST(amount AS DOUBLE))/1e6, 1) AS total_usd_m
FROM openalex.awards.helmsley_raw
GROUP BY recipient ORDER BY total_usd_m DESC LIMIT 20;


## Step 1.6: Fail-fast — Verify the Helmsley Funder Row Exists

**Runbook §2.2.4 mandatory.**

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320309446;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.helmsley_awards
USING delta
AS
WITH
helmsley_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320309446
),

raw_prepared AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        TRY_CAST(award_year AS INT) AS parsed_year,
        TRY_TO_DATE(award_date_iso, 'yyyy-MM-dd') AS parsed_start_date
    FROM openalex.awards.helmsley_raw
    WHERE recipient IS NOT NULL
      AND TRIM(recipient) != ''
      AND funder_award_id IS NOT NULL
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,

        -- display_name = the project_title (the actual purpose); fallback to recipient name
        COALESCE(NULLIF(TRIM(g.project_title), ''), g.recipient) as display_name,

        g.project_title as description,

        f.funder_id,
        g.funder_award_id,

        g.parsed_amount as amount,
        'USD' as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'grant' as funding_type,

        -- funder_scheme = the program (e.g., "Health Sciences", "Type 1 Diabetes")
        COALESCE(NULLIF(TRIM(g.program), ''), 'Helmsley Charitable Trust Grant') as funder_scheme,

        'helmsley_grants' as provenance,

        g.parsed_start_date as start_date,
        CAST(NULL AS DATE) as end_date,
        g.parsed_year as start_year,
        CAST(NULL AS INT) as end_year,

        -- lead_investigator: Helmsley grants are to organizations, not individuals.
        -- Per HHMI/Hewlett precedent (org-level recipient), put the recipient in
        -- the affiliation slot and leave given/family names NULL.
        struct(
            CAST(NULL AS STRING) as given_name,
            CAST(NULL AS STRING) as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.recipient), '') as name,
                'US' as country,  -- some Helmsley grants are international (Sub-Saharan Africa, Israel); a more rigorous mapping would parse city from the listing pages
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        g.landing_page_url as landing_page_url,

        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN helmsley_funder f
)

SELECT * FROM awards_transformed;


## Step 3: Insert Into `openalex_awards_raw` With Priority 112

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'helmsley_grants' AND priority = 112;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    112 as priority  -- Helmsley priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.helmsley_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.helmsley_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.helmsley_awards;


In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(landing_page_url) as has_url,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) as pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) as pct_start_date
FROM openalex.awards.helmsley_awards;


In [ ]:
%sql
SELECT funder_award_id, display_name, start_year, funder_scheme, amount, currency,
       lead_investigator.affiliation.name AS recipient_org,
       landing_page_url
FROM openalex.awards.helmsley_awards LIMIT 10;


In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.helmsley_awards
GROUP BY funder.display_name;


In [ ]:
%sql
SELECT start_year, COUNT(*) as grants, ROUND(SUM(amount)/1e6,1) as total_usd_m
FROM openalex.awards.helmsley_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC;


In [ ]:
%sql
-- §6.7 Amount/currency check. Expected: ~100% pct_amount, USD only.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_amount_nonzero,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt,
    ROUND(AVG(CASE WHEN amount > 0 THEN amount END), 0) AS avg_nonzero_amt,
    ROUND(SUM(amount) / 1e9, 2) AS total_usd_billions
FROM openalex.awards.helmsley_awards;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount)/1e6,1) as total_usd_m
FROM openalex.awards.helmsley_awards
GROUP BY funder_scheme ORDER BY total_usd_m DESC LIMIT 20;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.helmsley_awards;
